# Predict the pedestal using EPEDNN-SC loop for SPARC baseline scenario

In [1]:
import os
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sys
from pathlib import Path
import tempfile
import urllib.request # needed for geqdsk import
ROOT = Path.cwd().parent.parent.parent
sys.path.insert(0, str(ROOT))
from src.profiles_loop_solve import profiles_loop_solve

tokamaker_python_path = os.getenv('OFT_ROOTPATH')
if tokamaker_python_path is not None:
    sys.path.append(os.path.join(tokamaker_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.TokaMaker import TokaMaker
from OpenFUSIONToolkit.TokaMaker.meshing import load_gs_mesh
from OpenFUSIONToolkit.TokaMaker.util import create_isoflux, read_eqdsk

Authorization required, but no authorization protocol specified
Authorization required, but no authorization protocol specified


In [2]:
# Import the SPARC baseline scenario geqdsk (freely available from SPARCPublic)
import urllib.parse

_SPARC_PRD = 'https://raw.githubusercontent.com/cfs-energy/SPARCPublic/main/PrimaryReferenceDischarge'
_SPARC_GEQDSK = '2 - SPARC_DN_PRD_freegs_20221013'


def fetch_text(filename):
    """Read a file from the SPARCPublic PRD folder over HTTP."""
    url = f'{_SPARC_PRD}/{urllib.parse.quote(filename)}'
    print(f'Fetching {filename} from SPARCPublic...', end=' ', flush=True)
    with urllib.request.urlopen(url) as resp:
        text = resp.read().decode()
    print('done.')
    return text


# Persist gEQDSK for mhd_load / profiles_loop_solve (needs a filepath).
# Leading 'g' keeps equil_tag = Path(mhd_fp).name[1:] sensible.
mhd_fp = 'gSPARC_DN_PRD.geqdsk'
geqdsk_text = fetch_text(_SPARC_GEQDSK)
Path(mhd_fp).write_text(geqdsk_text)
print(f'Wrote {mhd_fp}')

eqdsk = read_eqdsk(mhd_fp)

print(f"  Ip      = {eqdsk['ip']/1.E6:.2f} MA")
print(f"  F0      = {eqdsk['rcentr']*eqdsk['bcentr']:.2f} T.m  (B0 = {eqdsk['rcentr']*eqdsk['bcentr']/1.85:.2f} T at R0 = 1.85 m)")
print(f"  p_axis  = {eqdsk['pres'][0]/1.E6:.2f} MPa")
print(f"  axis    = ({eqdsk['raxis']:.3f}, {eqdsk['zaxis']:.3f}) m")


Fetching 2 - SPARC_DN_PRD_freegs_20221013 from SPARCPublic... done.
Wrote gSPARC_DN_PRD.geqdsk
  Ip      = 8.70 MA
  F0      = 22.49 T.m  (B0 = 12.16 T at R0 = 1.85 m)
  p_axis  = 2.60 MPa
  axis    = (1.890, -0.000) m


In [8]:
# Import SPARC kinetic profiles from local SPARCPublic exports.
# TRANSP: 5 - transp_20221013.txt
# CGYRO:  6 - cgyro_20221013.txt
# Sections: rho, polflux, q, te [keV], ti [keV], ne [10^19/m^3] (ni optional).

TRANSP_PATH = Path('5 - transp_20221013.txt')
CGYRO_PATH = Path('6 - cgyro_20221013.txt')
PROFILE_SOURCE = 'cgyro'   # 'transp' or 'cgyro' — feeds manual_profs for the solve
OVERPLOT_PROFILES = False   # if True, plot TRANSP and CGYRO together for comparison


def read_sparcpublic_profiles(path):
    """
    Parse a SPARCPublic-style profile text file (TRANSP or CGYRO layout).

    Format: blocks headed by ``# name | unit``, each followed by
    ``index  value`` rows.

    Returns
    -------
    dict
        Keys for available quantities, each an ndarray. Always includes
        ``rho``. Electron/ion profiles added when present:
          - ne  [m^-3]
          - Te  [keV]
          - Ti  [keV]   (if present)
          - ni  [m^-3]  (if present)
        Also keeps raw auxiliaries ``polflux``, ``q`` when present, plus
        ``units`` metadata.
    """
    path = Path(path)
    sections = {}
    current = None
    current_unit = ''
    values = []

    with open(path) as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith('#'):
                if current is not None:
                    sections[current] = {
                        'data': np.asarray(values, dtype=float),
                        'unit': current_unit,
                    }
                # "# te | keV"  or  "# rho | -"
                body = line[1:].strip()
                name, _, unit = body.partition('|')
                current = name.strip().lower()
                current_unit = unit.strip()
                values = []
            else:
                parts = line.split()
                if len(parts) < 2:
                    continue
                values.append(float(parts[-1]))

    if current is not None:
        sections[current] = {
            'data': np.asarray(values, dtype=float),
            'unit': current_unit,
        }

    if 'rho' not in sections:
        raise KeyError(f'No rho grid found in {path}. Sections: {list(sections)}')

    rho = sections['rho']['data']
    out = {
        'rho': rho,
        'units': {'rho': sections['rho']['unit']},
    }
    for key in ('polflux', 'q'):
        if key in sections:
            out[key] = sections[key]['data']
            out['units'][key] = sections[key]['unit']

    # Temperatures: file stores keV
    for src, dst in (('te', 'Te'), ('ti', 'Ti')):
        if src in sections:
            if len(sections[src]['data']) != len(rho):
                raise ValueError(
                    f'{src} length {len(sections[src]["data"])} != rho length {len(rho)}'
                )
            out[dst] = sections[src]['data'].copy()
            out['units'][dst] = 'keV'

    # Densities: convert known file units -> m^-3
    def _density_to_si(name, unit, data):
        u = unit.lower().replace(' ', '')
        if '10^20' in u or '10**20' in u or '1e20' in u:
            return data * 1e20
        if '10^19' in u or '10**19' in u or '1e19' in u:
            return data * 1e19
        if u in ('m^-3', 'm-3', '1/m^3', '/m^3'):
            return data.copy()
        raise ValueError(f'Unrecognized {name} unit {unit!r} in {path}')

    for src, dst in (('ne', 'ne'), ('ni', 'ni')):
        if src in sections:
            if len(sections[src]['data']) != len(rho):
                raise ValueError(
                    f'{src} length {len(sections[src]["data"])} != rho length {len(rho)}'
                )
            out[dst] = _density_to_si(src, sections[src]['unit'], sections[src]['data'])
            out['units'][dst] = 'm^-3'

    return out


read_transp_profiles = read_sparcpublic_profiles
read_cgyro_profiles = read_sparcpublic_profiles


def psi_n_and_rho_psi(profiles):
    """Return (psi_N, rho_psi=sqrt(psi_N)) from polflux."""
    if 'polflux' not in profiles:
        raise KeyError('Profile file needs polflux to map rho -> psi_N correctly')
    pf = profiles['polflux']
    psi_N = (pf - pf[0]) / (pf[-1] - pf[0])
    rho_psi = np.sqrt(np.clip(psi_N, 0.0, None))
    return psi_N, rho_psi


def build_manual_profs(profiles):
    """manual_profs for kprof_loc='manual rho grid' with psi_N-consistent rho."""
    _, rho_psi = psi_n_and_rho_psi(profiles)
    manual_profs = {
        'Te': profiles['Te'],
        'rho_Te': rho_psi,
        'ne': profiles['ne'] / 1e20,   # m^-3 -> 10^20 m^-3 (solver multiplies by 1e20)
        'rho_ne': rho_psi,
    }
    if 'Ti' in profiles:
        manual_profs['Ti'] = profiles['Ti']
        manual_profs['rho_Ti'] = rho_psi
    if 'ni' in profiles:
        manual_profs['ni'] = profiles['ni'] / 1e20
        manual_profs['rho_ni'] = rho_psi
    return manual_profs


def calc_pressure_profile(profiles):
    """
    Thermal pressure from kinetic profiles on the polflux psi_N grid.

    Defaults to ``p = 2 * ne * Te`` (Te = Ti, ne = ni). If ``Ti`` is present,
    uses ``p = ne * Te + ni * Ti``, with ``ni = ne`` when ``ni`` is missing.

    Parameters
    ----------
    profiles : dict
        Output of ``read_sparcpublic_profiles`` (ne [m^-3], Te/Ti [keV]).

    Returns
    -------
    psi_N : ndarray
    p : ndarray
        Pressure in Pa.
    mode : str
        Formula used.
    """
    eV_to_J = 1.602176634e-19
    psi_N, _ = psi_n_and_rho_psi(profiles)
    ne = np.asarray(profiles['ne'], dtype=float)
    Te_J = np.asarray(profiles['Te'], dtype=float) * 1e3 * eV_to_J  # keV -> J

    if 'Ti' in profiles:
        Ti_J = np.asarray(profiles['Ti'], dtype=float) * 1e3 * eV_to_J
        if 'ni' in profiles:
            ni = np.asarray(profiles['ni'], dtype=float)
            mode = 'ne*Te + ni*Ti'
        else:
            ni = ne
            mode = 'ne*Te + ne*Ti (ni=ne)'
        p = ne * Te_J + ni * Ti_J
    else:
        p = 2.0 * ne * Te_J
        mode = '2*ne*Te'

    return psi_N, p, mode


def _print_profile_summary(label, path, profiles):
    psi_N, rho_psi = psi_n_and_rho_psi(profiles)
    print(f'Loaded {path.name} ({label}): {len(profiles["rho"])} points')
    print(f'  quantities: {[k for k in profiles if k != "units"]}')
    print(f'  psi_N mapping: using polflux -> rho_psi=sqrt(psi_N) '
          f'(max |rho_file^2 - psi_N| = {np.max(np.abs(profiles["rho"]**2 - psi_N)):.3f})')
    print(f'  Te axis  = {profiles["Te"][0]:.2f} keV,  edge = {profiles["Te"][-1]:.2f} keV')
    print(f'  ne axis  = {profiles["ne"][0]/1e20:.2f}e20 m^-3,  '
          f'edge = {profiles["ne"][-1]/1e20:.2f}e20 m^-3')
    if 'Ti' in profiles:
        print(f'  Ti axis  = {profiles["Ti"][0]:.2f} keV,  edge = {profiles["Ti"][-1]:.2f} keV')
    else:
        print('  Ti: not in file')
    if 'ni' in profiles:
        print(f'  ni axis  = {profiles["ni"][0]/1e20:.2f}e20 m^-3')
    else:
        print('  ni: not in file')


profiles_transp = read_transp_profiles(TRANSP_PATH)
profiles_cgyro = read_cgyro_profiles(CGYRO_PATH)

source = PROFILE_SOURCE.strip().lower()
if source == 'transp':
    profiles = profiles_transp
    prof_path = TRANSP_PATH
elif source == 'cgyro':
    profiles = profiles_cgyro
    prof_path = CGYRO_PATH
else:
    raise ValueError(f"PROFILE_SOURCE must be 'transp' or 'cgyro', got {PROFILE_SOURCE!r}")

manual_profs = build_manual_profs(profiles)

print(f'Using {prof_path.name} ({source}) for manual_profs')
_print_profile_summary(source, prof_path, profiles)

if OVERPLOT_PROFILES:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
    for ax, qty, ylabel in zip(
        axes,
        ('Te', 'ne'),
        (r'$T_e$ (keV)', r'$n_e$ ($10^{20}$ m$^{-3}$)'),
    ):
        for label, prof, color in (
            ('TRANSP', profiles_transp, 'C0'),
            ('CGYRO', profiles_cgyro, 'C1'),
        ):
            psi_N, _ = psi_n_and_rho_psi(prof)
            y = prof[qty] if qty == 'Te' else prof[qty] / 1e20
            ax.plot(psi_N, y, '-', color=color, label=label)
        ax.set_xlabel(r'$\psi_N$', fontsize=13)
        ax.set_ylabel(ylabel, fontsize=13)
        ax.legend(fontsize=11)
        ax.tick_params(labelsize=11)
        ax.set_xlim(0.9, 1)
    axes[0].set_ylim(0, 8)
    axes[1].set_ylim(0, 4)
    axes[0].set_title('Electron temperature')
    axes[1].set_title('Electron density')
    fig.suptitle('TRANSP vs CGYRO profiles', fontsize=14)
    plt.tight_layout()
    plt.show()


Using 6 - cgyro_20221013.txt (cgyro) for manual_profs
Loaded 6 - cgyro_20221013.txt (cgyro): 201 points
  quantities: ['rho', 'polflux', 'q', 'Te', 'Ti', 'ne']
  psi_N mapping: using polflux -> rho_psi=sqrt(psi_N) (max |rho_file^2 - psi_N| = 0.202)
  Te axis  = 23.42 keV,  edge = 0.27 keV
  ne axis  = 4.96e20 m^-3,  edge = 0.85e20 m^-3
  Ti axis  = 17.13 keV,  edge = 0.58 keV
  ni: not in file


In [4]:
# Import heating power and Zeff from SPARCPublic POPCON table
POPCON_PATH = Path('1 - PRD_POPCON_20221013.csv')


def read_popcon(path, keys):
    """
    Read named entries from the SPARCPublic POPCON CSV.

    Returns
    -------
    dict
        key -> {'value': float, 'unit': str}
    """
    import csv

    wanted = {k.lower(): k for k in keys}
    found = {}
    with open(path, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = (row.get('Variable') or row.get('variable') or '').strip()
            if name.lower() not in wanted:
                continue
            val = float(row.get('Value') or row.get('value'))
            unit = (row.get('Unit') or row.get('unit') or '').strip()
            found[wanted[name.lower()]] = {'value': val, 'unit': unit}

    missing = [k for k in keys if k not in found]
    if missing:
        raise KeyError(f'Missing {missing} in {path}')
    return found


def _to_watts(value, unit):
    unit_to_W = {'': 1.0, 'w': 1.0, 'kw': 1e3, 'mw': 1e6, 'gw': 1e9}
    key = (unit or 'W').strip().lower()
    if key not in unit_to_W:
        raise ValueError(f'Unrecognized power unit {unit!r}')
    return value * unit_to_W[key]


popcon = read_popcon(POPCON_PATH, keys=('Ohmic power', 'RF power', 'Zeff'))

P_ohmic = _to_watts(popcon['Ohmic power']['value'], popcon['Ohmic power']['unit'])  # W
P_RF = _to_watts(popcon['RF power']['value'], popcon['RF power']['unit'])            # W
Zeff = float(popcon['Zeff']['value'])  # dimensionless

# P_tot_e: heating power to electrons [W].
# Saarelma et al. take ~half of the total (Ohmic+RF) heating as electron channel.
P_tot_e = (P_ohmic + P_RF) / 2

print(f'Loaded POPCON inputs from {POPCON_PATH.name}:')
print(f'  P_ohmic = {P_ohmic/1e6:.2f} MW')
print(f'  P_RF    = {P_RF/1e6:.2f} MW')
print(f'  P_tot_e = {P_tot_e/1e6:.2f} MW  (= 0.5*(P_ohmic+P_RF), in W: {P_tot_e:.3e})')
print(f'  Zeff    = {Zeff:.3g}')


Loaded POPCON inputs from 1 - PRD_POPCON_20221013.csv:
  P_ohmic = 1.70 MW
  P_RF    = 11.10 MW
  P_tot_e = 6.40 MW  (= 0.5*(P_ohmic+P_RF), in W: 6.400e+06)
  Zeff    = 1.5


In [5]:
# Parameters for model #

# Output directory
output_dir = 'SPARC_output_logs'
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Scan parameters
x_res = 50
free_params = {
    'alpha_crit': 10,
    'C_KBM': 0.1,
    'De_chie_etg': 1,
    'nFC_x0': 1e15,
    'ncx_x0_ratio': 17.8
}
eped_tol_max = 1e-5
eped_iter_max = 5
kbm_treatment = "picard"
kbm_gate_eps = 0.1
picard_gate_mode = "average"
picard_max_it = 50
picard_rtol = 1e-8
picard_relax = 1.0

verbose_EPEDNNloop = False
verbose_sc = False


In [7]:
# Model run
ped_wid, ped_h_out = profiles_loop_solve(
    MHD_FP = mhd_fp,
    kprof_loc = 'manual rho grid',
    manual_profs = manual_profs,
    P_tot_e = P_tot_e,
    species = 'D-T',
    Z_i = Zeff,
    out_dir = output_dir,
    x_res = x_res,
    free_params = free_params,
    eped_tol_max = eped_tol_max,
    eped_iter_max = eped_iter_max,
    kbm_gate_eps = kbm_gate_eps,
    kbm_treatment = kbm_treatment,
    picard_gate_mode = picard_gate_mode,
    picard_max_it = picard_max_it,
    picard_rtol = picard_rtol,
    picard_relax = picard_relax,
    ig = 'manual',
    epednn_model = 'EPED_SPARC',
    verbose = verbose_EPEDNNloop,
    verbose_sc = verbose_sc,
)


Setting up EPEDNN...
bt: [12.49263273]
Base model built.
EPEDNN-SC Loop Iter 0
betan: [1.14878909]
{'a': 0.5698608945, 'betan': 1.1487890927555913, 'bt': 12.492632729938999, 'delta': 0.5261178927939227, 'ip': 8.7, 'kappa': 1.954842479544804, 'm': 2.5, 'neped': 30.00016387565005, 'r': 1.8508492314999998, 'zeffped': 1.5}
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 522ms/step
[4.3137726e+02 5.2432518e-02]
Pedestal height: 0.43137726187705994 MPa, Pedestal width: 0.05243251845240593 (psi_N)
Setting up EPEDNN...
bt: [12.49263273]
EPEDNN-SC Loop Iter 1
betan: [1.13469651]
{'a': 0.5698608945, 'betan': 1.1346965122929213, 'bt': 12.492632729938999, 'delta': 0.5261178927939227, 'ip': 8.7, 'kappa': 1.954842479544804, 'm': 2.5, 'neped': 22.62199211866225, 'r': 1.8508492314999998, 'zeffped': 1.5}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step
[3.8972342e+02 4.9575157e-02]
Pedestal height: 0.38972342014312744 MPa, Pedestal width: 0.04957515746355057 (psi_N)
Normalized pedestal pressure height and width tolerance: -0.151056

In [11]:
# Profile pressure at the EPEDNN-predicted pedestal top (psi_N = 1 - ped_wid)
psi_N_p, p_Pa, p_mode = calc_pressure_profile(profiles)
psi_ped_top = 1.0 - float(ped_wid)
p_at_ped_top = float(np.interp(psi_ped_top, psi_N_p, p_Pa))

print(f'Pressure formula: {p_mode}')
print(f'EPEDNN ped_top at psi_N = 1 - ped_wid = {psi_ped_top:.6f}')
print(f'Profile p(psi_ped_top) = {p_at_ped_top/1e3:.3f} kPa  ({p_at_ped_top/1e6:.4f} MPa)')
print(f'EPEDNN ped_h_out       = {float(ped_h_out)*1e3:.3f} kPa  ({float(ped_h_out):.4f} MPa)')
print(f'Accuracy of EPEDNN prediction: {100*(float(p_at_ped_top/1e6) - float(ped_h_out))/float(ped_h_out):.2f}%')


Pressure formula: ne*Te + ne*Ti (ni=ne)
EPEDNN ped_top at psi_N = 1 - ped_wid = 0.951154
Profile p(psi_ped_top) = 404.295 kPa  (0.4043 MPa)
EPEDNN ped_h_out       = 377.602 kPa  (0.3776 MPa)
Accuracy of EPEDNN prediction: 7.07%


In [ ]:
# Save model output
out_dict = {
    'mhd_fp': mhd_fp,
    'profiles': manual_profs,
    'ESCAPE_ped_h': ped_h_out,
    'ESCAPE_ped_wid': ped_wid,
    'experimental_ped_h': p_at_ped_top,
}
np.save(output_dir+'/SPARC_pedestal_prediction.npy', out_dict)